# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print('Pop-Tart-like rows:', len(pop))
pop

Pop-Tart-like rows: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
straw = pop[pop['product_name'].str.lower().str.contains('straw', na=False)]
print(straw[['sku', 'product_name']].to_string(index=False))

print('\nPop-Tart rows that are NOT strawberry:')
print(pop.drop(straw.index)[['sku', 'product_name']].to_string(index=False))

          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry

Pop-Tart rows that are NOT strawberry:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
n_before = len(cat)

CANON = {s: 'POPTART-STRAW' for s in straw['sku']}
cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])   # fillna keeps unmapped SKUs as themselves
cat[['sku', 'canonical_sku']].head(10)

,sku,canonical_sku
0,PT-12,POPTART-STRAW
1,SKU#459812,POPTART-STRAW
2,POPTART-STRAW,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print('BEFORE:'); print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()
print('\nAFTER strip + title:'); print(cat['category_clean'].value_counts(dropna=False))

# Same thing under two labels -> merge by hand
cat['category_clean'] = cat['category_clean'].replace({'Breakfast Foods': 'Breakfast'})
print('\nAFTER merging duplicates:'); print(cat['category_clean'].value_counts(dropna=False))

BEFORE:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

AFTER strip + title:
category_clean
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

AFTER merging duplicates:
category_clean
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
import re

def name_key(s):
    words = re.sub(r'[^a-z0-9 ]', ' ', str(s).lower()).split()
    return ' '.join(sorted(w.rstrip('s') for w in words))   # rstrip('s') folds Tart/Tarts

cat['name_key'] = cat['product_name'].map(name_key)
n_skus = cat.groupby('name_key')['sku'].nunique()
multi = n_skus[n_skus > 1]
print(f'{len(multi)} candidate groups')
cat[cat['name_key'].isin(multi.index)].sort_values('name_key')[['name_key', 'sku', 'product_name']]

1 candidate groups


,name_key,sku,product_name
0,pop strawberry tart,PT-12,pop-tarts strawberry
1,pop strawberry tart,SKU#459812,Strawberry Pop-Tart
2,pop strawberry tart,POPTART-STRAW,Pop-Tarts Strawberry


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert cat['canonical_sku'].notna().all(), 'mapping produced nulls'
assert len(cat) == n_before, f'row count changed: {n_before} -> {len(cat)}'
print('mapping is safe.')

mapping is safe.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.


In my run of the catalog (which fell back to the 3-row sample), the only fragmented family was strawberry Pop-Tarts, so I could not identify two other families from this data. From Wednesday’s studio sample, which showed the same kinds of problems the midterm data has, bottled water appears fragmented: WATER-24 and BTL-WATER-24 are the same 24-pack under two SKUs, and I would map both to WATER-24. A second likely family is flashlights, which appeared there under a single SKU (FLASH-AA) but would need to be checked for variants in the full catalog. On the full file I would find these by running the Q5 name-key grouping and reviewing every group with more than one SKU.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Consolidation stops being data cleaning when the question moves from “are these spelled differently?” to “are these the same thing to a buyer?” PT-12, SKU#459812 and POPTART-STRAW are clearly one product, which is a cleaning problem. Blueberry vs. strawberry Pop-Tarts is a business decision: merging them would average away the strawberry-specific signal the analysis is about, while keeping them apart fragments a family someone might want to see as a whole. I am genuinely unsure about case or size variants such as a 12-pack vs. a 24-pack of the same item, which could be one product or two with different demand. I would ask the category manager or whoever owns the sales question, because the right grouping depends on what decision the analysis supports.